# 01 · What's New: A Tour of Python 3.10 → 3.15

**Module 17 – What's New in Python** · Estimated time: 60 min

> The 📗 bootcamp was written for Python ~3.6–3.8 and the 📘 handbook touches 3.10–3.13. This notebook is a **one-stop tour of every important change since 3.10**, each with a runnable example and a link to the lesson that covers it in depth.

| Version | Released | Status (Oct 2026) |
|---|---|---|
| 3.10 | Oct 2021 | security fixes only, **end-of-life Oct 2026** |
| 3.11 | Oct 2022 | security fixes only |
| 3.12 | Oct 2023 | security fixes only |
| 3.13 | Oct 2024 | moving from bug fixes to security-only |
| 3.14 | Oct 2025 | bug fixes (current mainstream) |
| **3.15** | **9 Oct 2026** | **newest release** |

> 💡 Use a version that's still supported. For new projects in late 2026: **3.14 or 3.15**. Check https://devguide.python.org/versions/

In [1]:
import sys
print("Running on Python", sys.version.split()[0])

Running on Python 3.15.0


---
## 🐍 Python 3.10 (2021)

### Structural pattern matching: `match`/`case` → Module 04 · 02

In [2]:
def http(status):
    match status:
        case 200 | 201:
            return "OK"
        case 404:
            return "Not found"
        case int(code) if code >= 500:
            return "Server error"
        case _:
            return "Other"
print([http(s) for s in (200, 404, 503, 302)])

['OK', 'Not found', 'Server error', 'Other']


### `X | Y` union types → Module 05 · 06

In [3]:
def parse(value: int | str) -> int | None:
    return int(value) if str(value).isdigit() else None
print(parse("42"), parse("x"), isinstance(5, int | str))

42 None True


### Parenthesised context managers, `zip(strict=True)`, `int.bit_count()`, `itertools.pairwise`, much better error messages

In [4]:
from itertools import pairwise
print(list(zip([1, 2], "ab", strict=True)), (255).bit_count(), list(pairwise("abc")))

[(1, 'a'), (2, 'b')] 8 [('a', 'b'), ('b', 'c')]


---
## 🐍 Python 3.11 (2022): "Faster CPython"

- **10–60% faster** than 3.10 (adaptive specialising interpreter)
- **Fine-grained error locations** in tracebacks (`^^^^` under the exact expression)

### Exception groups, `except*` and notes → Module 07 · 01

In [5]:
try:
    raise ExceptionGroup("batch failed", [ValueError("bad row 3"), KeyError("id")])
except* ValueError as eg:
    print("value errors:", eg.exceptions)
except* KeyError as eg:
    print("key errors:", eg.exceptions)

err = ValueError("invalid date")
err.add_note("while importing orders.csv line 17")
print(err.__notes__)

value errors: (ValueError('bad row 3'),)
key errors: (KeyError('id'),)
['while importing orders.csv line 17']


### `tomllib`, `asyncio.TaskGroup`, `asyncio.timeout`, `typing.Self`, `enum.StrEnum`, `datetime.UTC`, `contextlib.chdir`

In [6]:
import tomllib, asyncio
from enum import StrEnum, auto
from datetime import datetime, UTC

print(tomllib.loads('name = "app"\nport = 8080'))

class Color(StrEnum):
    RED = auto()
print(Color.RED == "red", datetime.now(UTC).tzinfo)

async def main():
    async with asyncio.TaskGroup() as tg:
        t = tg.create_task(asyncio.sleep(0.1, result="done"))
    return t.result()
print(await main())

{'name': 'app', 'port': 8080}
True UTC
done


---
## 🐍 Python 3.12 (2023)

### f-strings formalised (PEP 701): same quotes, multi-line, comments → Module 02 · 05

In [7]:
names = ["Asha", "Ravi"]
print(f"Team: {", ".join(names)}")

Team: Asha, Ravi


### New generic syntax & `type` aliases (PEP 695) → Module 05 · 06

In [8]:
type Pair[T] = tuple[T, T]

def swap[T](p: Pair[T]) -> Pair[T]:
    return (p[1], p[0])

print(swap((1, 2)))

(2, 1)


### `@typing.override`, `itertools.batched`, `Path.walk()`, per-interpreter GIL, clearer errors ("Did you forget to import 'sys'?")

In [9]:
from itertools import batched
from typing import override

class Base:
    def greet(self): return "hi"

class Child(Base):
    @override                       # type checkers verify that Base really has greet()
    def greet(self): return "hello"

print(list(batched(range(7), 3)), Child().greet())

[(0, 1, 2), (3, 4, 5), (6,)] hello


⚠️ **3.12 removals/deprecations:** `distutils` removed (use setuptools/pyproject), `asynchat`/`asyncore`/`imp`/`smtpd` removed, `datetime.utcnow()` deprecated.

---
## 🐍 Python 3.13 (2024)

- **New interactive REPL**: colours, multi-line editing, `exit` without parentheses
- **Experimental free-threaded build** (no GIL) and **experimental JIT compiler** → Module 13 · 08
- **"Dead batteries" removed** (PEP 594): `cgi`, `cgitb`, `crypt`, `telnetlib`, `nntplib`, `imghdr`, `sndhdr`, `pipes`, `uu`, `xdrlib`, `aifc`, `audioop`, `chunk`, `mailcap`, `msilib`, `nis`, `ossaudiodev`, `spwd`, `sunau`, and `lib2to3`

### `copy.replace()` and `@warnings.deprecated`

In [10]:
import copy, warnings
from dataclasses import dataclass

@dataclass(frozen=True)
class Point:
    x: int
    y: int

print(copy.replace(Point(1, 2), y=99))

@warnings.deprecated("use new_api()")
def old_api(): return 1
print(old_api.__deprecated__)

Point(x=1, y=99)
use new_api()


---
## 🐍 Python 3.14 (2025): "π-thon"

### Template strings: `t"..."` (PEP 750) → Module 02 · 05

In [11]:
from string.templatelib import Interpolation
user = "<b>hacker</b>"
tmpl = t"Hello {user}!"
print(type(tmpl).__name__, [type(p).__name__ for p in tmpl])
print("".join(p if isinstance(p, str) else str(p.value).replace("<", "&lt;") for p in tmpl))

Template ['str', 'Interpolation', 'str']
Hello &lt;b>hacker&lt;/b>!


### Deferred evaluation of annotations (PEP 649/749): forward references just work → Module 05 · 06

In [12]:
class Tree:
    def add(self, child: Tree) -> Tree:     # no quotes needed any more
        return self
print(Tree.add.__annotations__)

{'child': <class '__main__.Tree'>, 'return': <class '__main__.Tree'>}


### `except` without parentheses (PEP 758), `finally` + `return` warning (PEP 765) → Module 07 · 01

In [13]:
try:
    int("nope")
except ValueError, TypeError:
    print("caught")

caught


### Standard library goodies
- `compression.zstd` (Zstandard) → Module 11 · 04
- `concurrent.interpreters` + `InterpreterPoolExecutor` → Module 13 · 08
- Free-threaded build **officially supported** (PEP 779)
- `pathlib.Path.copy()/.move()` → Module 11 · 02
- `python -m pdb -p PID` attaches to a running process → Module 07 · 02
- `map(..., strict=True)`, `uuid.uuid7()`, `heapq.heapify_max()`, `date.strptime()`, `float.from_number()`
- Syntax highlighting in the REPL; colourful `unittest`, `argparse` and `json` CLI output

In [14]:
import uuid, heapq
from datetime import date
print(list(map(pow, [2, 3], [3, 2], strict=True)))
print(uuid.uuid7())                          # time-ordered UUIDs: great as database keys
h = [3, 1, 4, 1, 5]
heapq.heapify_max(h)
print(heapq.heappop_max(h), date.strptime("2026-10-20", "%Y-%m-%d"))

[8, 9]
01a159ab-8653-7438-95f8-6f662ef4398c
5 2026-10-20


---
## 🐍 Python 3.15 (October 2026): newest

### Explicit lazy imports (PEP 810) → Module 08 · 01

```python
lazy import json                 # loaded only when json is first used
lazy from pathlib import Path
```
Only at module level; errors appear on first use; global switch `-X lazy_imports=all`.

### `frozendict` built-in (PEP 814) → Module 03 · 04

In [15]:
cfg = frozendict(host="localhost", port=8080)
print(cfg, hash(cfg) == hash(frozendict(port=8080, host="localhost")))

frozendict({'host': 'localhost', 'port': 8080}) True


### `sentinel` built-in (PEP 661) → Module 05 · 02

In [16]:
MISSING = sentinel("MISSING")
def get(key, default=MISSING):
    return "no default given" if default is MISSING else default
print(get("a"), get("a", None))

no default given None


### Unpacking in comprehensions (PEP 798) → Module 03 · 05

In [17]:
print([*row for row in [[1, 2], [3], [4, 5]]], {**d for d in [{"a": 1}, {"b": 2}]})

[1, 2, 3, 4, 5] {'a': 1, 'b': 2}


### UTF-8 is the default text encoding (PEP 686) → Module 11 · 01
`open("file.txt")` now uses UTF-8 everywhere (Windows included). Opt out with `PYTHONUTF8=0` or `encoding="locale"`.

### New `profiling` package with the Tachyon sampling profiler (PEP 799) → Module 12 · 05
```bash
python -m profiling.sampling run --flamegraph -o profile.html app.py
python -m profiling.tracing app.py          # cProfile's new home
```

### Other 3.15 highlights
- **JIT compiler** significantly upgraded (~7–12% faster on supported platforms)
- **Free-threading installed by default** with the macOS installers; stable ABI for free-threaded extensions
- Typing: `TypedDict` with typed extra items (PEP 728), `TypeForm` (PEP 747), disjoint bases (PEP 800)
- `re.prefixmatch()` (clearer name for `re.match()`) → Module 12 · 04
- Package startup configuration files (PEP 829); frame pointers enabled by default (better profiling)
- Better `AttributeError` suggestions and more colour in CLI output
- (Observed in this course's builds) CPython 3.15 caches small integers from **-5 to 1024** (previously -5 to 256), which is another reason never to compare numbers with `is` (Module 02 · 08)

In [18]:
import re
print(re.prefixmatch(r"\d+", "2026 is here"))

<re.Match object; span=(0, 4), match='2026'>


⚠️ **3.15 removals & behaviour changes:** keyword-argument `NamedTuple("P", x=int)` removed; `TypedDict("TD")` with no fields removed; `strptime` with `%d` but no year raises `ValueError`; `profile` module deprecated (removal in 3.17).

---
## How to keep up
- *What's New* pages: https://docs.python.org/3/whatsnew/
- Run your test suite with `python -W error::DeprecationWarning` to surface deprecations early
- Tools like **Ruff** (`UP` rules / pyupgrade) automatically modernise old syntax

## 🏋️ Practice
1. Pick three features from this tour you didn't know and write a small example of each.
2. Run `python -W error::DeprecationWarning -c "import datetime; datetime.datetime.utcnow()"` and read the error.
3. Run `ruff check --select UP your_old_code.py` on a script from the original bootcamp and see what it suggests.